# Chapitre 2 : Reproductibilité et développement professionnel

*MLSMM2151 Data Science · Séance 1, vendredi 13 novembre, matin · Corentin Vande Kerckhove*

> **CONSEIL** : commencez par enregistrer une copie de ce notebook dans votre Google Drive
> (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes.

## Notebook 3 sur 3 : démarrer le devoir 1

**Durée estimée : 45 minutes.**

### Objectifs

À la fin de ce notebook, vous saurez :

- lancer votre variante individuelle du devoir, et dire ce qu'elle change ;
- décrire la forme d'un fichier de 540 000 lignes en trois commandes ;
- transformer une demande vague en question précise, par écrit ;
- chiffrer un problème de qualité et en tirer une décision provisoire ;
- pousser des commits dont les messages racontent votre démarche.

**À la fin de ce notebook, votre dépôt de devoir contiendra deux commits de
plus, et votre devoir aura commencé.**

## Comment travailler pendant ces 45 minutes

Vous allez travailler avec **deux onglets** ouverts dans Colab :

- **celui-ci**, qui donne les consignes et montre chaque geste sur les
  annonces Airbnb. Il ne se complète pas : ses cellules de code tournent telles
  quelles ;
- **votre `notebook.ipynb`**, ouvert depuis votre dépôt de devoir
  (`Fichier` > `Ouvrir un notebook` > onglet `GitHub`). C'est là que tout
  votre travail s'écrit, sur les données Online Retail.

Le principe est toujours le même : je montre un geste sur Airbnb, vous le
refaites sur vos données. Quatre blocs, chacun se termine par un résultat
visible, et deux d'entre eux par un commit.

**Avant de commencer**, vérifiez l'état attendu après le notebook précédent :
l'invitation à votre dépôt `devoir-1-...` est acceptée, `notebook.ipynb` est
ouvert dans l'autre onglet, et l'onglet `Commits` de votre dépôt sur github.com
montre votre commit « Renseigne mon nom et mon matricule ». Si ce n'est pas le
cas, levez la main maintenant : le tableau de dépannage du corrigé du
notebook 2 couvre les cas habituels.

In [ ]:
# Ce notebook travaille sur les annonces Airbnb, jamais sur vos données de devoir.
!curl -fsSL -o donnees.py https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/utils/donnees.py

import pandas as pd
from donnees import charger

annonces = charger("airbnb_raw")

## Bloc 1 : votre variante, et la forme de vos données

### Ce que fait `mon_sujet`

Votre dépôt contient un fichier `sujet.py`. Sa fonction `mon_sujet` prend votre
matricule et en tire, par une empreinte `sha256`, **une période d'analyse et un
pays de référence** qui vous sont propres. Pour le matricule fictif
`12345678`, elle renvoie :

```
periode_debut            2011-01-01
periode_fin              2011-04-30
pays_reference           France
matricule                12345678
graine                   6916
```

Deux conséquences. Vos chiffres ne seront pas ceux de votre voisin, et c'est
voulu : comparer vos résultats est utile, les copier ne l'est pas. Et le
correcteur rejoue `mon_sujet` avec votre matricule : un devoir qui ne travaille
pas sur sa propre variante perd deux points.

### Regarder la forme

Avant toute analyse, trois gestes, et toujours dans cet ordre : combien de
lignes et de colonnes, quels types, et à quoi ressemblent quelques lignes
**tirées au hasard**. Pas les premières : un fichier de transactions est trié
par date, et `head()` ne vous montrerait que le premier jour.

In [ ]:
print(f"{annonces.shape[0]} lignes, {annonces.shape[1]} colonnes")
annonces.dtypes

In [ ]:
# random_state fixe le tirage : vous et votre voisin verrez les mêmes lignes.
annonces.sample(5, random_state=0)

### Hack-Time 🛠️

Vous venez de voir la forme des annonces Airbnb en trois commandes. Dans
l'autre onglet, dans votre `notebook.ipynb` :

1. Remplacez `VOTRE_MATRICULE` par votre numéro UCLouvain et exécutez la
   cellule. Notez votre période d'analyse et votre pays de référence : ils
   reviendront aux parties 3 et 4.
2. Exécutez la cellule de chargement. Le fichier pèse 48 Mo, comptez une
   minute. Une piste : si vous avez monté votre Google Drive avant, il ne sera
   téléchargé qu'une fois.
3. Dans une nouvelle cellule, affichez la forme, les types, et cinq lignes
   tirées au hasard : `transactions.shape`, `transactions.dtypes`,
   `transactions.sample(5, random_state=0)`.
4. Comparez avec le codebook : quelle colonne n'a pas le type que vous
   attendiez, et pourquoi ?

**Huit minutes.**

**Défi optionnel**
`Customer_ID` est un identifiant. Pourquoi pandas en fait-il un nombre à
virgule ? Qu'est-ce que cela vous apprend sur la colonne avant même de
compter quoi que ce soit ?

*Vos notes :* la forme obtenue, et le type qui vous a surpris.

## Bloc 2 : cadrer le problème

Souvenez-vous de l'étape 1 du cycle de vie d'un projet, vu en Data
Analytics : la demande arrive
vague, et le premier travail est d'en faire une question précise. L'énoncé du
devoir exige trois choses, et la grille les note une par une : **la décision**
que votre travail doit permettre de prendre, ce que « base client propre »
veut dire **ici**, et ce que le fichier **ne permettra pas** de faire.

### Un cadrage complet, sur Airbnb

Imaginons la demande suivante, adressée au data scientist d'une société qui
gère des logements à Bruxelles : « On aimerait savoir à quel prix mettre un
nouveau logement en ligne. »

> **La décision.** Quand un propriétaire nous confie un logement, nous devons
> fixer son prix d'affichage initial, avant qu'il n'ait le moindre avis. La
> question est donc : *quel prix par nuit afficher pour un logement dont on
> connaît la commune, le type, la capacité et les équipements, mais pas
> encore l'historique ?*
>
> **Ce que « prix » veut dire ici.** Le prix par nuit affiché sur la
> plateforme à la date du snapshot (21 juin 2025), hors frais de ménage et de
> service, pour les 6 721 annonces de la ville. Nous écartons les annonces
> sans prix affiché, et nous bornons l'étude aux prix entre 20 et 1 000 euros,
> au-delà desquels il s'agit d'erreurs de saisie ou de logements hors de notre
> marché.
>
> **Ce que ces données ne permettent pas.** Elles ne disent rien de la
> **saisonnalité** : un seul jour est photographié, pas une année. Elles
> donnent le prix **affiché**, pas le prix **payé** après remises et
> négociations. Et l'occupation est **estimée** par Inside Airbnb à partir des
> avis, elle n'est pas observée. Nous pourrons proposer un prix d'affichage
> cohérent avec le marché un jour de juin ; nous ne pourrons pas promettre un
> revenu.

### Ce qui distingue un cadrage utile

| Cadrage faible | Cadrage utile |
|---|---|
| Reformule la demande avec d'autres mots | Nomme **une décision** et qui la prend |
| « Les clients » | Dit **ce qu'est une ligne** : un client, une commande, une visite ? |
| « Sur les données disponibles » | Fixe **une période**, et dit pourquoi celle-là |
| Promet tout | Écrit **un refus** : ce que les données ne diront pas, et pourquoi |
| S'adresse au professeur | S'adresse à la personne qui a posé la question |

### Hack-Time 🛠️

Vous venez de lire un cadrage complet sur la question Airbnb. À votre tour,
sur la demande de la direction commerciale, dans la cellule « *Votre
analyse* » de la partie 1 de votre notebook :

1. Écrivez **la décision** que votre travail doit permettre de prendre, en
   une phrase qui commence par « La direction pourra décider... ». Une
   piste : relisez les mots de la demande, « campagnes de l'année prochaine »,
   et demandez-vous ce qu'une campagne a besoin de savoir sur un client.
2. Écrivez ce que « base client propre » veut dire **ici** : une ligne par
   quoi, sur quelle période, avec quelles informations au minimum.
3. Écrivez **un refus** : une chose que la direction pourrait espérer de ce
   fichier et qu'il ne permettra pas. Une piste : regardez ce que les huit
   colonnes ne contiennent pas.
4. `Fichier` > `Enregistrer une copie dans GitHub`, avec un message qui
   complète « ce commit va... », par exemple `Rédige une première version
   du cadrage`. Pas le message proposé par défaut.

**Dix minutes.** Une demi-page, en phrases. C'est une première version :
vous la réécrirez après la séance 2, et c'est précisément ce que l'historique
de commits doit montrer.

**Défi optionnel**
Relisez votre cadrage en vous mettant à la place de la direction : contient-il
un mot qu'elle ne comprendrait pas ? Si oui, c'est à réécrire. La partie 6 du
devoir s'adresse à elle, autant prendre le pli maintenant.

*Vos notes :* ce qui vous a résisté dans le cadrage.

## Bloc 3 : un premier diagnostic chiffré

La partie 2 du devoir vous demande un diagnostic de qualité. Un diagnostic,
ce n'est pas « il y a des manquants ». C'est un geste en quatre temps :

1. **compter** les lignes concernées ;
2. **rapporter** ce nombre au total, en proportion ;
3. **croiser** avec une autre colonne, pour savoir si ces lignes sont des
   lignes comme les autres ou une population à part ;
4. **écrire une décision provisoire**, en une phrase.

Voici le geste sur les annonces Airbnb dont le prix manque.

In [ ]:
manque_prix = annonces["price"].isna()
print(f"{manque_prix.sum()} annonces sans prix, soit {manque_prix.mean():.1%}")

In [ ]:
# Les annonces sans prix sont-elles des annonces comme les autres ?
# On compare deux colonnes qui n'ont rien à voir avec le prix.
annonces.groupby(manque_prix)[["number_of_reviews", "availability_365"]].median()

**Lecture.** 1 020 annonces, soit 15,2 %, n'ont pas de prix. Ce n'est pas
aléatoire : leur disponibilité médiane sur les 365 prochains jours est de
**0 jour**, contre 217 pour les autres, et elles ont moins d'avis. Ce sont des
annonces **inactives**, retirées du calendrier, dont le prix n'est plus
affiché. Elles ne décrivent pas le marché d'aujourd'hui.

**Constater n'est pas décider.** La décision provisoire tient en une phrase :
*« J'écarte ces annonces pour toute question de prix, mais je les garde pour
compter les logements par commune, où elles comptent encore. »* Provisoire,
parce qu'on la rediscutera quand on connaîtra mieux le fichier. Mais écrite,
parce qu'une décision non écrite devient vite une suppression silencieuse que
personne ne retrouve.

### Hack-Time 🛠️

Nous venons de chiffrer les prix manquants d'Airbnb, puis de vérifier si ces
annonces ressemblaient aux autres. Refaites exactement ce geste sur les
identifiants clients manquants, dans la partie 2 de votre notebook :

1. Comptez les lignes sans `Customer_ID`, en nombre et en proportion. Une
   piste : `sans_client = transactions["Customer_ID"].isna()`, puis `.sum()`
   et `.mean()`. Vous devez retrouver l'ordre de grandeur annoncé par
   l'énoncé.
2. Croisez avec une autre colonne pour savoir si ces lignes sont des lignes
   comme les autres. Une piste :
   `transactions.groupby(sans_client)[["Quantity", "Price"]].median()`, ou
   `transactions.groupby(sans_client)["Country"].agg(lambda s: s.mode()[0])`.
3. Écrivez trois phrases dans la cellule « *Votre analyse* » : combien, ce
   que cela veut dire sur le plan métier, et ce que vous décidez
   **provisoirement**. La décision peut être « je ne sais pas encore, voici
   ce qu'il me faut pour trancher ».
4. `Fichier` > `Enregistrer une copie dans GitHub`, message du type
   `Chiffre les identifiants clients manquants`.

**Douze minutes.**

**Défi optionnel**
Calculez le chiffre d'affaires, `Quantity * Price`, des lignes sans client,
rapporté au chiffre d'affaires total. L'énoncé parle d'un quart des lignes :
est-ce aussi un quart du chiffre d'affaires ? Puis demandez-vous ce que cela
change pour une base **client** : ces lignes peuvent-elles y entrer ? Peut-on
pour autant les oublier dans la conclusion ?

*Vos notes :* votre décision provisoire, et ce qu'il vous manque pour la
confirmer.

## Bloc 4 : vérifier, et commencer `REFLEXION.md`

Vous avez poussé deux commits depuis Colab. Avant de fermer, deux
vérifications sur github.com, et un fichier à ouvrir.

**Votre dépôt raconte-t-il ce que vous avez fait ?** L'onglet `Commits` de
votre dépôt est ce que verra le correcteur en premier. Chaque commit doit
porter un message lisible, et le diff de chaque commit doit correspondre au
message.

**Aucun fichier de données n'est entré.** La racine du dépôt ne doit montrer
aucun `.csv`. Le `.gitignore` du squelette l'interdit, mais vérifier prend
cinq secondes.

**`REFLEXION.md` s'écrit au fil du devoir, pas la veille.** C'est le fichier
que tout le monde oublie, et un devoir rendu sans lui n'est pas corrigé. Il se
modifie sans Colab : sur github.com, ouvrez le fichier, cliquez sur le crayon,
écrivez, puis `Commit changes`.

**Et la suite ?** Les parties 2 à 5 du devoir s'appuient sur les chapitres 3
et 4, que nous verrons à la séance 2, le 27 novembre. D'ici là, vous pouvez
relire votre cadrage, explorer le fichier, et chercher le problème de qualité
« que vous avez trouvé vous-même » exigé par la partie 2.

### Hack-Time 🛠️

Vous avez poussé deux commits. Avant de fermer, vérifions que votre dépôt
raconte bien ce que vous avez fait, et commençons le fichier que tout le
monde oublie.

1. Sur github.com, ouvrez votre dépôt, onglet `Commits`. Vous devez voir
   quatre commits : le squelette, votre nom, le cadrage, le diagnostic.
   Cliquez sur le dernier et regardez le diff.
2. Vérifiez qu'aucun fichier `.csv` n'apparaît à la racine du dépôt.
3. Ouvrez `REFLEXION.md`, cliquez sur le crayon, renseignez votre nom et votre
   matricule, puis écrivez deux phrases dans « Ma démarche » : ce que vous avez
   fait aujourd'hui, et ce qui vous a surpris. `Commit changes`, directement
   sur `main`, avec un vrai message.
4. En dessous, dans la même section, notez quelles parties du devoir attendent
   la séance 2, et ce que vous pouvez faire d'ici là.

**Huit minutes.**

**Défi optionnel**
Dans l'onglet `Commits`, chaque commit porte une heure. Imaginez votre
historique le 1er décembre : qu'est-ce qu'un correcteur en conclut si les
cinq commits sont espacés de quatre minutes ? Fixez-vous dès maintenant deux
dates de travail avant le 27 novembre, et écrivez-les dans `REFLEXION.md`.

*Vos notes :* vos deux dates de travail.

## À retenir

- Votre variante est tirée de votre matricule par `mon_sujet`, et le
  correcteur la rejoue : travaillez sur la vôtre.
- Un diagnostic, c'est un nombre, une proportion, un croisement, et une
  décision provisoire écrite. Constater n'est pas décider.
- Un cadrage nomme une décision, définit ce qu'est une ligne, fixe une
  période, et écrit un refus.
- Un commit par geste, avec un message qui complète « ce commit va... ».
  L'historique fait partie du rendu.
- `REFLEXION.md` s'écrit au fil du devoir, pas la veille de l'échéance.

**La suite** : la séance 2, le 27 novembre, outille les parties 2 à 5 du
devoir : explorer, diagnostiquer, traiter le temps, construire la base client,
encoder. D'ici là, relisez votre cadrage et poussez au moins un commit.